# Curadoria e Construção do Subconjunto Final

Este notebook constrói o subconjunto definitivo de pinturas utilizado nos experimentos do TCC.

A seleção parte exclusivamente das obras aprovadas pela auditoria de integridade realizada na etapa anterior.

Os principais objetivos são:

- definir uma semente aleatória fixa;
- selecionar as obras de forma determinística e reproduzível;
- balancear as quatro macroclasses;
- preservar a composição dos estilos originais que formam as macroclasses agregadas;
- criar identificadores estáveis;
- incorporar os metadados relevantes;
- produzir um manifesto final em CSV.

As imagens originais não serão armazenadas no repositório. O manifesto deverá permitir recuperar cada obra a partir do dataset WikiArt.

## 1. Configuração do ambiente e parâmetros

A semente do experimento é fixada em `42`.

O valor da semente não possui significado estatístico especial. Sua função é garantir que a mesma regra de seleção produza sempre o mesmo subconjunto quando aplicada aos mesmos dados de entrada.

As definições das macroclasses permanecem centralizadas em `src/data/labels.py`.

In [18]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from datasets import load_dataset


DATASET_ID = "huggan/wikiart"

SEED = 42
N_POR_CLASSE = 2500
TOTAL_ESPERADO = 10_000


def encontrar_raiz_projeto(inicio: Path = Path.cwd()) -> Path:
    """Localiza a raiz do projeto procurando pela pasta src/."""
    for caminho in [inicio, *inicio.parents]:
        if (caminho / "src").is_dir():
            return caminho

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto a partir do diretório atual."
    )


RAIZ_PROJETO = encontrar_raiz_projeto()

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))


from src.data.labels import CLASSES_FINAIS


PASTA_METADATA = RAIZ_PROJETO / "data" / "metadata"

CAMINHO_AUDITORIA = PASTA_METADATA / "auditoria_integridade.csv"
CAMINHO_MANIFESTO = PASTA_METADATA / "dataset_final.csv"

print(f"Raiz do projeto: {RAIZ_PROJETO}")
print(f"Dataset: {DATASET_ID}")
print(f"SEED: {SEED}")
print(f"Obras por macroclasse: {N_POR_CLASSE:,}")
print(f"Total final esperado: {TOTAL_ESPERADO:,}")

Raiz do projeto: c:\Users\Brum_\OneDrive\Documentos 1\TCC\Wikiart
Dataset: huggan/wikiart
SEED: 42
Obras por macroclasse: 2,500
Total final esperado: 10,000


## 2. Carregamento das obras aprovadas pela auditoria

O arquivo `auditoria_integridade.csv` é a fonte de entrada desta etapa.

Somente registros com `Aprovada_Auditoria == True` podem participar da seleção. Dessa forma, obras descartadas por duplicação ou por algum problema técnico não retornam ao experimento.

In [19]:
df_auditoria = pd.read_csv(CAMINHO_AUDITORIA)

colunas_obrigatorias_auditoria = {
    "Indice_Dataset",
    "Estilo",
    "Macro_Estilo",
    "Aprovada_Auditoria",
    "Hash_SHA256_Pixels",
    "Largura",
    "Altura",
    "Total_Pixels",
    "Megapixels",
    "Proporcao",
    "Orientacao",
}

faltantes = (
    colunas_obrigatorias_auditoria
    - set(df_auditoria.columns)
)

if faltantes:
    raise ValueError(
        "O arquivo de auditoria não contém todas as colunas esperadas: "
        f"{sorted(faltantes)}"
    )


df_elegiveis = (
    df_auditoria[
        df_auditoria["Aprovada_Auditoria"] == True
    ]
    .copy()
)

quantidade_por_classe = (
    df_elegiveis["Macro_Estilo"]
    .value_counts()
    .reindex(CLASSES_FINAIS)
)

print(f"Registros na auditoria: {len(df_auditoria):,}")
print(f"Obras elegíveis: {len(df_elegiveis):,}")

display(
    quantidade_por_classe
    .rename("Quantidade_Aprovada")
    .to_frame()
)

Registros na auditoria: 26,426
Obras elegíveis: 26,276


,Quantidade_Aprovada
Macro_Estilo,
Renaissance,6532
Baroque,4193
Impressionism,12993
Cubism,2558


In [20]:
classes_insuficientes = quantidade_por_classe[
    quantidade_por_classe < N_POR_CLASSE
]

if not classes_insuficientes.empty:
    raise RuntimeError(
        "Há macroclasses com menos obras elegíveis do que o necessário:\n"
        f"{classes_insuficientes}"
    )

### Resultado de referência

Após a auditoria, permaneceram elegíveis:

- Renaissance: 6.532 obras;
- Baroque: 4.193 obras;
- Impressionism: 12.993 obras;
- Cubism: 2.558 obras.

Como todas as macroclasses possuem pelo menos 2.500 registros válidos, foi mantida a meta de **2.500 pinturas por classe**, totalizando **10.000 obras**.

## 3. Distribuição dos estilos originais

Baroque e Impressionism correspondem diretamente a um único estilo original.

Renaissance e Cubism são macroclasses agregadas. Para evitar alterar artificialmente sua composição interna, a seleção preserva aproximadamente a proporção dos estilos originais disponíveis após a auditoria.

In [21]:
distribuicao_estilos_elegiveis = (
    df_elegiveis
    .groupby(["Macro_Estilo", "Estilo"])
    .size()
    .reset_index(name="Quantidade")
    .sort_values(["Macro_Estilo", "Estilo"])
)

display(distribuicao_estilos_elegiveis)

,Macro_Estilo,Estilo,Quantidade
0,Baroque,Baroque,4193
1,Cubism,Analytical_Cubism,110
2,Cubism,Cubism,2232
3,Cubism,Synthetic_Cubism,216
4,Impressionism,Impressionism,12993
5,Renaissance,Early_Renaissance,1385
6,Renaissance,High_Renaissance,1333
7,Renaissance,Mannerism_Late_Renaissance,1276
8,Renaissance,Northern_Renaissance,2538


## 4. Alocação proporcional por estilo original

A quantidade destinada a cada estilo é calculada pelo **método dos maiores restos**:

1. calcula-se a quota proporcional exata;
2. utiliza-se inicialmente a parte inteira da quota;
3. as vagas restantes são atribuídas aos maiores restos.

Esse procedimento garante que a soma final seja exatamente 2.500 por macroclasse, sem distorções causadas por arredondamentos independentes.

In [22]:
def calcular_alocacao_proporcional(
    df_classe,
    total_desejado,
):
    """
    Distribui total_desejado proporcionalmente entre os estilos
    originais da macroclasse usando o método dos maiores restos.
    """
    contagens = (
        df_classe["Estilo"]
        .value_counts()
        .sort_index()
    )

    proporcoes = contagens / contagens.sum()
    quotas = proporcoes * total_desejado

    alocacao = np.floor(quotas).astype(int)

    restante = (
        total_desejado
        - int(alocacao.sum())
    )

    restos = (
        quotas - alocacao
    ).sort_values(
        ascending=False,
        kind="stable",
    )

    for estilo in restos.index[:restante]:
        alocacao.loc[estilo] += 1

    resultado = pd.DataFrame({
        "Disponivel": contagens,
        "Proporcao_Original": proporcoes,
        "Quota_Exata": quotas,
        "Selecionar": alocacao,
    })

    resultado["Proporcao_Original_Pct"] = (
        resultado["Proporcao_Original"] * 100
    ).round(2)

    return resultado

In [23]:
alocacoes = {}

for macroclasse in CLASSES_FINAIS:
    df_classe = df_elegiveis[
        df_elegiveis["Macro_Estilo"] == macroclasse
    ]

    alocacao = calcular_alocacao_proporcional(
        df_classe,
        N_POR_CLASSE,
    )

    if int(alocacao["Selecionar"].sum()) != N_POR_CLASSE:
        raise RuntimeError(
            f"Alocação inválida para {macroclasse}."
        )

    alocacoes[macroclasse] = alocacao

    print(f"\n{macroclasse}")
    display(
        alocacao[
            [
                "Disponivel",
                "Proporcao_Original_Pct",
                "Selecionar",
            ]
        ]
    )


Renaissance


,Disponivel,Proporcao_Original_Pct,Selecionar
Estilo,,,
Early_Renaissance,1385,21.20,530
High_Renaissance,1333,20.41,510
Mannerism_Late_Renaissance,1276,19.53,488
Northern_Renaissance,2538,38.85,972



Baroque


,Disponivel,Proporcao_Original_Pct,Selecionar
Estilo,,,
Baroque,4193,100.0,2500



Impressionism


,Disponivel,Proporcao_Original_Pct,Selecionar
Estilo,,,
Impressionism,12993,100.0,2500



Cubism


,Disponivel,Proporcao_Original_Pct,Selecionar
Estilo,,,
Analytical_Cubism,110,4.30,108
Cubism,2232,87.26,2181
Synthetic_Cubism,216,8.44,211


### Alocação de referência

A seleção final preserva a seguinte composição:

**Renaissance**

- Early Renaissance: 530;
- High Renaissance: 510;
- Mannerism / Late Renaissance: 488;
- Northern Renaissance: 972.

**Cubism**

- Analytical Cubism: 108;
- Cubism: 2.181;
- Synthetic Cubism: 211.

Os subestilos não são artificialmente equalizados.

## 5. Seleção determinística das pinturas

A seleção não utiliza `pandas.sample()`.

Cada obra recebe uma chave SHA-256 calculada a partir de uma semente derivada e do `Indice_Dataset`. Dentro de cada estrato, as obras são ordenadas por essa chave e são mantidas as primeiras posições necessárias.

A estratégia garante que:

- os mesmos dados de entrada;
- a mesma ordem de macroclasses e estilos;
- a mesma `SEED`;
- e o mesmo código

produzam exatamente a mesma seleção.

O `hash()` nativo do Python não é utilizado porque não é adequado como identificador determinístico persistente entre execuções.

In [24]:
def chave_selecao(indice_dataset, seed):
    """Gera uma chave SHA-256 determinística para ordenar candidatos."""
    texto = f"{seed}:{int(indice_dataset)}"

    return hashlib.sha256(
        texto.encode("utf-8")
    ).hexdigest()


def selecionar_macroclasse(
    df,
    macroclasse,
    alocacao,
    seed,
):
    """
    Seleciona deterministicamente uma macroclasse preservando
    a alocação proporcional de seus estilos originais.
    """
    partes = []

    estilos = sorted(
        alocacao.index.tolist()
    )

    for numero_estilo, estilo in enumerate(estilos):
        quantidade = int(
            alocacao.loc[
                estilo,
                "Selecionar",
            ]
        )

        candidatos = (
            df[
                (df["Macro_Estilo"] == macroclasse)
                & (df["Estilo"] == estilo)
            ]
            .copy()
        )

        seed_estrato = (
            seed + numero_estilo
        )

        candidatos["Chave_Selecao"] = (
            candidatos["Indice_Dataset"]
            .apply(
                lambda indice: chave_selecao(
                    indice,
                    seed_estrato,
                )
            )
        )

        selecionados = (
            candidatos
            .sort_values(
                [
                    "Chave_Selecao",
                    "Indice_Dataset",
                ]
            )
            .head(quantidade)
            .copy()
        )

        if len(selecionados) != quantidade:
            raise RuntimeError(
                f"Não foi possível selecionar {quantidade} obras "
                f"do estrato {macroclasse} / {estilo}."
            )

        partes.append(selecionados)

    return pd.concat(
        partes,
        ignore_index=True,
    )

In [25]:
partes_selecionadas = []

for numero_classe, macroclasse in enumerate(
    CLASSES_FINAIS
):
    selecionados_classe = selecionar_macroclasse(
        df=df_elegiveis,
        macroclasse=macroclasse,
        alocacao=alocacoes[macroclasse],
        seed=SEED + (numero_classe * 100),
    )

    partes_selecionadas.append(
        selecionados_classe
    )


df_selecionado = pd.concat(
    partes_selecionadas,
    ignore_index=True,
)

df_selecionado = (
    df_selecionado
    .sort_values(
        [
            "Macro_Estilo",
            "Estilo",
            "Indice_Dataset",
        ]
    )
    .reset_index(drop=True)
)

df_selecionado = df_selecionado.drop(
    columns=["Chave_Selecao"],
    errors="ignore",
)

## 6. Validação da seleção

Antes de incorporar metadados adicionais, a seleção é verificada quanto ao tamanho, balanceamento, unicidade dos índices e composição interna dos estilos.

In [26]:
quantidade_final_por_classe = (
    df_selecionado["Macro_Estilo"]
    .value_counts()
    .reindex(CLASSES_FINAIS)
)

indices_duplicados = (
    df_selecionado["Indice_Dataset"]
    .duplicated()
    .sum()
)

if len(df_selecionado) != TOTAL_ESPERADO:
    raise RuntimeError(
        f"Total selecionado inesperado: {len(df_selecionado):,}"
    )

if not quantidade_final_por_classe.eq(N_POR_CLASSE).all():
    raise RuntimeError(
        "As macroclasses não possuem exatamente "
        f"{N_POR_CLASSE:,} obras cada."
    )

if indices_duplicados != 0:
    raise RuntimeError(
        f"Foram encontrados {indices_duplicados} índices duplicados."
    )


print(f"Total selecionado: {len(df_selecionado):,}")
print(f"Índices duplicados: {indices_duplicados}")

display(
    quantidade_final_por_classe
    .rename("Quantidade")
    .to_frame()
)

distribuicao_selecionada = (
    df_selecionado
    .groupby(["Macro_Estilo", "Estilo"])
    .size()
    .reset_index(name="Quantidade")
)

display(distribuicao_selecionada)

Total selecionado: 10,000
Índices duplicados: 0


,Quantidade
Macro_Estilo,
Renaissance,2500
Baroque,2500
Impressionism,2500
Cubism,2500


,Macro_Estilo,Estilo,Quantidade
0,Baroque,Baroque,2500
1,Cubism,Analytical_Cubism,108
2,Cubism,Cubism,2181
3,Cubism,Synthetic_Cubism,211
4,Impressionism,Impressionism,2500
5,Renaissance,Early_Renaissance,530
6,Renaissance,High_Renaissance,510
7,Renaissance,Mannerism_Late_Renaissance,488
8,Renaissance,Northern_Renaissance,972


## 7. Recuperação de artista e gênero

O manifesto deve preservar os principais metadados do WikiArt.

Como `df_selecionado` mantém o `Indice_Dataset`, os metadados de artista e gênero são recuperados exatamente para as 10.000 obras escolhidas.

A ordem fornecida a `Dataset.select()` é preservada e corresponde à ordem atual do DataFrame.

In [27]:
dataset = load_dataset(DATASET_ID)
train = dataset["train"]

if len(train) != 81_444:
    raise RuntimeError(
        "A quantidade de registros do WikiArt difere da execução de referência. "
        "Verifique a versão do dataset antes de continuar."
    )

artist_names = train.features["artist"].names
genre_names = train.features["genre"].names

indices_selecionados = (
    df_selecionado["Indice_Dataset"]
    .astype(int)
    .tolist()
)

dados_selecionados = train.select(
    indices_selecionados
)

df_selecionado["artist_id"] = (
    dados_selecionados["artist"]
)

df_selecionado["genre_id"] = (
    dados_selecionados["genre"]
)

df_selecionado["Artista"] = (
    df_selecionado["artist_id"]
    .map(lambda valor: artist_names[valor])
)

df_selecionado["Genero"] = (
    df_selecionado["genre_id"]
    .map(lambda valor: genre_names[valor])
)

display(
    df_selecionado[
        [
            "Indice_Dataset",
            "Estilo",
            "Macro_Estilo",
            "Artista",
            "Genero",
        ]
    ].head(10)
)

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/45 [00:00<?, ?it/s]

,Indice_Dataset,Estilo,Macro_Estilo,Artista,Genero
0,1,Baroque,Baroque,rembrandt,religious_painting
1,20,Baroque,Baroque,rembrandt,sketch_and_study
2,39,Baroque,Baroque,rembrandt,portrait
3,55,Baroque,Baroque,rembrandt,portrait
4,98,Baroque,Baroque,rembrandt,Unknown Genre
5,111,Baroque,Baroque,rembrandt,Unknown Genre
6,194,Baroque,Baroque,rembrandt,portrait
7,286,Baroque,Baroque,rembrandt,religious_painting
8,355,Baroque,Baroque,rembrandt,portrait
9,412,Baroque,Baroque,rembrandt,sketch_and_study


## 8. Identificador estável da pintura

O `Indice_Dataset` é mantido porque permite recuperar a obra na versão utilizada do WikiArt, mas sua estabilidade depende da organização dessa versão do dataset.

Por isso, o manifesto também utiliza o hash completo dos pixels auditados e cria um identificador legível:

`wikiart_<24 primeiros caracteres do SHA-256>`

Os 24 caracteres hexadecimais correspondem a 96 bits. A unicidade é verificada explicitamente no subconjunto final, e o SHA-256 completo continua armazenado no manifesto.

In [28]:
if df_selecionado["Hash_SHA256_Pixels"].isna().any():
    raise RuntimeError(
        "Existem obras selecionadas sem SHA-256 registrado."
    )

df_selecionado["Painting_ID"] = (
    "wikiart_"
    + df_selecionado["Hash_SHA256_Pixels"].str[:24]
)

ids_duplicados = (
    df_selecionado["Painting_ID"]
    .duplicated()
    .sum()
)

if ids_duplicados != 0:
    raise RuntimeError(
        f"Foram encontrados {ids_duplicados} Painting_ID duplicados."
    )

print(
    f"Painting_ID únicos: "
    f"{df_selecionado['Painting_ID'].nunique():,}"
)

Painting_ID únicos: 10,000


## 9. Verificação pós-seleção: gênero

O balanceamento foi realizado por macroclasse e, nas classes agregadas, por estilo original. A distribuição de gênero **não foi artificialmente equalizada**.

Esta análise verifica se a curadoria preservou os principais padrões temáticos observados na EDA.

In [29]:
genero_por_classe = (
    pd.crosstab(
        df_selecionado["Macro_Estilo"],
        df_selecionado["Genero"],
        normalize="index",
    )
    .mul(100)
    .reindex(CLASSES_FINAIS)
    .round(2)
)

display(genero_por_classe)

Genero,Unknown Genre,abstract_painting,cityscape,genre_painting,illustration,landscape,nude_painting,portrait,religious_painting,sketch_and_study,still_life
Macro_Estilo,,,,,,,,,,,
Renaissance,14.60,0.0,0.56,2.12,0.20,0.80,0.60,16.64,56.72,7.56,0.20
Baroque,21.32,0.0,1.20,18.12,0.20,4.04,0.88,22.48,23.96,4.88,2.92
Impressionism,11.28,0.0,12.00,20.00,0.04,33.68,2.88,14.56,0.20,2.16,3.20
Cubism,19.20,5.0,5.76,13.00,0.20,8.12,5.24,13.80,0.80,3.28,25.60


### Resultado de referência

A seleção preservou os principais padrões temáticos do conjunto candidato:

- Renaissance continua fortemente associado a `religious_painting`;
- Impressionism mantém forte presença de `landscape`;
- Cubism mantém concentração relevante de `still_life`;
- Baroque permanece mais heterogêneo.

O subconjunto balanceado, portanto, não eliminou as associações entre gênero e movimento já identificadas como possíveis fontes de *shortcut learning*.

## 10. Verificação pós-seleção: autoria

A autoria também é analisada como possível variável correlacionada com as classes.

`Unknown Artist` não deve ser interpretado como um único artista. O rótulo agrega um número indeterminado de autores cuja identidade não está disponível nos metadados.

Sua elevada frequência constitui principalmente uma **limitação para análises e controles por autoria**, e não um atalho diretamente acessível ao modelo, que receberá apenas as imagens.

In [30]:
print(
    "Rótulos distintos de artista:",
    df_selecionado["Artista"].nunique(),
)

display(
    df_selecionado["Artista"]
    .value_counts()
    .head(20)
    .rename("Quantidade")
    .to_frame()
)

Rótulos distintos de artista: 79


,Quantidade
Artista,
Unknown Artist,5178
rembrandt,429
pablo-picasso,303
pierre-auguste-renoir,287
albrecht-durer,264
claude-monet,250
juan-gris,188
georges-braque,153
peter-paul-rubens,148


In [31]:
artistas_por_classe = (
    df_selecionado
    .groupby(["Macro_Estilo", "Artista"])
    .size()
    .reset_index(name="Quantidade")
)

artistas_por_classe["Percentual_Classe"] = (
    artistas_por_classe
    .groupby("Macro_Estilo")["Quantidade"]
    .transform(
        lambda valores: valores / valores.sum() * 100
    )
    .round(2)
)

top_artistas_por_classe = (
    artistas_por_classe
    .sort_values(
        ["Macro_Estilo", "Quantidade"],
        ascending=[True, False],
    )
    .groupby(
        "Macro_Estilo",
        group_keys=False,
    )
    .head(10)
)

artistas_distintos_por_classe = (
    df_selecionado
    .groupby("Macro_Estilo")["Artista"]
    .nunique()
    .reindex(CLASSES_FINAIS)
)

display(top_artistas_por_classe)

display(
    artistas_distintos_por_classe
    .rename("Rotulos_Artista_Distintos")
    .to_frame()
)

,Macro_Estilo,Artista,Quantidade,Percentual_Classe
0,Baroque,Unknown Artist,1538,61.52
9,Baroque,rembrandt,429,17.16
8,Baroque,peter-paul-rubens,148,5.92
2,Baroque,bartolome-esteban-murillo,99,3.96
1,Baroque,anthony-van-dyck,98,3.92
4,Baroque,frans-hals,95,3.80
5,Baroque,jacob-jordaens,58,2.32
3,Baroque,canaletto,17,0.68
6,Baroque,joseph-wright,9,0.36
7,Baroque,joshua-reynolds,9,0.36


,Rotulos_Artista_Distintos
Macro_Estilo,
Renaissance,15
Baroque,10
Impressionism,50
Cubism,13


### Interpretação da autoria

Na execução de referência, foram observados 79 rótulos distintos de artista.

O rótulo `Unknown Artist` representa mais da metade do subconjunto, mas não pode ser tratado como evidência de concentração em um único autor. Ele pode agregar obras produzidas por diversos autores não identificados.

Entre os autores conhecidos, permanecem concentrações relevantes em artistas historicamente associados às respectivas classes, como Rembrandt, Picasso, Renoir, Monet e Albrecht Dürer.

A seleção não impõe balanceamento por autor. A possibilidade de características específicas de determinados artistas contribuírem para as previsões deverá ser considerada na avaliação dos modelos.

## 11. Construção do manifesto final

O manifesto preserva apenas informações necessárias para identificação, rastreabilidade e análises posteriores.

As imagens não são incorporadas ao CSV e não são versionadas no GitHub.

In [32]:
COLUNAS_MANIFESTO = [
    "Painting_ID",
    "Indice_Dataset",
    "Hash_SHA256_Pixels",
    "Estilo",
    "Macro_Estilo",
    "artist_id",
    "Artista",
    "genre_id",
    "Genero",
    "Largura",
    "Altura",
    "Total_Pixels",
    "Megapixels",
    "Proporcao",
    "Orientacao",
]

df_manifesto = (
    df_selecionado[
        COLUNAS_MANIFESTO
    ]
    .copy()
)

## 12. Validações finais

Antes de persistir o manifesto, são verificados:

- total de 10.000 obras;
- 10.000 `Painting_ID` únicos;
- 10.000 índices originais únicos;
- exatamente 2.500 pinturas por macroclasse;
- ausência de valores nulos nas colunas do manifesto.

Os rótulos textuais `Unknown Artist` e `Unknown Genre` não são valores ausentes e permanecem registrados normalmente.

In [33]:
if len(df_manifesto) != TOTAL_ESPERADO:
    raise RuntimeError(
        f"Manifesto possui {len(df_manifesto):,} linhas; "
        f"esperado: {TOTAL_ESPERADO:,}."
    )

if df_manifesto["Painting_ID"].nunique() != TOTAL_ESPERADO:
    raise RuntimeError(
        "Painting_ID não é único no manifesto."
    )

if df_manifesto["Indice_Dataset"].nunique() != TOTAL_ESPERADO:
    raise RuntimeError(
        "Indice_Dataset não é único no manifesto."
    )

contagem_manifesto = (
    df_manifesto["Macro_Estilo"]
    .value_counts()
    .reindex(CLASSES_FINAIS)
)

if not contagem_manifesto.eq(N_POR_CLASSE).all():
    raise RuntimeError(
        "O manifesto não está balanceado em 2.500 obras por macroclasse."
    )

ausencias = (
    df_manifesto
    .isna()
    .sum()
    .rename("Valores_Ausentes")
    .to_frame()
)

display(ausencias)

if int(ausencias["Valores_Ausentes"].sum()) != 0:
    raise RuntimeError(
        "O manifesto contém valores ausentes inesperados."
    )

,Valores_Ausentes
Painting_ID,0
Indice_Dataset,0
Hash_SHA256_Pixels,0
Estilo,0
Macro_Estilo,0
artist_id,0
Artista,0
genre_id,0
Genero,0
Largura,0


## 13. Persistência e teste de leitura

O manifesto definitivo é salvo em:

`data/metadata/dataset_final.csv`

Após a escrita, o arquivo é lido novamente e validado para confirmar que o artefato persistido contém as 10.000 pinturas esperadas.

In [34]:
df_manifesto.to_csv(
    CAMINHO_MANIFESTO,
    index=False,
    encoding="utf-8",
)

df_teste_manifesto = pd.read_csv(
    CAMINHO_MANIFESTO
)

if len(df_teste_manifesto) != TOTAL_ESPERADO:
    raise RuntimeError(
        "O CSV persistido não possui a quantidade esperada de registros."
    )

if df_teste_manifesto["Painting_ID"].nunique() != TOTAL_ESPERADO:
    raise RuntimeError(
        "O CSV persistido contém Painting_ID duplicados."
    )

print(
    f"Manifesto salvo em: {CAMINHO_MANIFESTO}"
)
print(
    f"Registros recuperados do CSV: "
    f"{len(df_teste_manifesto):,}"
)

display(df_teste_manifesto.head())

Manifesto salvo em: c:\Users\Brum_\OneDrive\Documentos 1\TCC\Wikiart\data\metadata\dataset_final.csv
Registros recuperados do CSV: 10,000


,Painting_ID,Indice_Dataset,Hash_SHA256_Pixels,Estilo,Macro_Estilo,artist_id,Artista,genre_id,Genero,Largura,Altura,Total_Pixels,Megapixels,Proporcao,Orientacao
0,wikiart_c16e8f4d7f4bbbad8ec5a985,1,c16e8f4d7f4bbbad8ec5a9855c3b3216cc437d3843d364...,Baroque,Baroque,20,rembrandt,7,religious_painting,1382,1659,2292738,2.29,0.833,Vertical
1,wikiart_1f4fc7187eb078304b26b218,20,1f4fc7187eb078304b26b21862be23ef72cdf81036d244...,Baroque,Baroque,20,rembrandt,8,sketch_and_study,1382,1664,2299648,2.30,0.831,Vertical
2,wikiart_cf43808a982719bfe58c8088,39,cf43808a982719bfe58c80889e0765d0049a29901899b9...,Baroque,Baroque,20,rembrandt,6,portrait,1382,1725,2383950,2.38,0.801,Vertical
3,wikiart_f9c0fde8a3d400f037ab6aff,55,f9c0fde8a3d400f037ab6affdc790437b65a911ca26f5b...,Baroque,Baroque,20,rembrandt,6,portrait,1382,1675,2314850,2.31,0.825,Vertical
4,wikiart_6ebd371fc5e8fb56317e92fc,98,6ebd371fc5e8fb56317e92fcd335fd45b3e5520ec97f84...,Baroque,Baroque,20,rembrandt,10,Unknown Genre,1382,1718,2374276,2.37,0.804,Vertical
